# Proyecto 3 — Optimización de Pricing en Rutas de Ferry

## Notebook 3 — Elasticidad precio-demanda

En este notebook analizamos la **elasticidad precio-demanda** para una compañía ficticia de transporte marítimo: **Levante Ferries**.

El objetivo es entender cómo responde la demanda ante cambios de precio y usar esa información para apoyar decisiones de pricing:

- Cuándo podría tener sentido subir precio.
- Cuándo conviene mantener precio.
- Cuándo puede ser recomendable promocionar.
- Qué rutas son más sensibles al precio.
- Qué rutas parecen más inelásticas.
- Qué impacto tienen distintos escenarios de precio sobre revenue y margen.

Este notebook parte del dataset creado en el Notebook 1 y analizado en el Notebook 2.

## 1. Objetivo del notebook

La elasticidad precio-demanda mide cuánto cambia la demanda cuando cambia el precio.

En términos sencillos:

- Si la demanda cae mucho cuando sube el precio, la ruta es **elástica**.
- Si la demanda cambia poco cuando sube el precio, la ruta es **inelástica**.
- Si una ruta tiene alta ocupación, buen margen y baja elasticidad, puede existir oportunidad de subida controlada de precio.
- Si una ruta tiene baja ocupación y alta elasticidad, puede existir oportunidad promocional.

En este notebook veremos tres niveles de análisis:

1. Elasticidad simple por ruta.
2. Elasticidad ajustada con variables de control.
3. Simulación de escenarios de precio.

In [ ]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# Configuración de pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Configuración visual
sns.set_theme(style="whitegrid")

print("Librerías importadas correctamente.")

In [ ]:
# ============================================================
# 2. CONEXIÓN CON GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive conectado correctamente.")
except:
    print("No estás ejecutando este notebook en Google Colab o Drive ya está montado.")

In [ ]:
# ============================================================
# 3. DEFINICIÓN DE RUTAS DEL PROYECTO
# ============================================================

base_path = "/content/drive/MyDrive/7 Colab Notebooks/1 Porfolio/Balearia/3 Pricing"

data_path = f"{base_path}/data/processed/ferry_pricing_dataset.csv"
reports_path = f"{base_path}/reports"
images_path = f"{base_path}/images"

os.makedirs(reports_path, exist_ok=True)
os.makedirs(images_path, exist_ok=True)

print("Ruta base del proyecto:")
print(base_path)

print("\nRuta del dataset:")
print(data_path)

## 2. Funciones auxiliares

Incluimos funciones para:

- Asegurar columnas numéricas.
- Exportar tablas en CSV estándar, CSV compatible con Excel España y XLSX.
- Clasificar elasticidad.
- Interpretar implicaciones de negocio.

In [ ]:
# ============================================================
# 4. FUNCIÓN AUXILIAR: ASEGURAR COLUMNAS NUMÉRICAS
# ============================================================

def force_numeric_columns(dataframe, columns):
    """
    Convierte columnas a formato numérico aunque vengan como texto.
    Soporta valores tipo:
    - 1234.56
    - 1234,56
    - 1.234,56 €
    - 45,5 %
    """
    
    df_copy = dataframe.copy()
    
    for col in columns:
        if col not in df_copy.columns:
            continue
        
        direct_conversion = pd.to_numeric(df_copy[col], errors="coerce")
        
        if direct_conversion.notna().mean() >= 0.80:
            df_copy[col] = direct_conversion
        else:
            df_copy[col] = (
                df_copy[col]
                .astype(str)
                .str.replace("€", "", regex=False)
                .str.replace("%", "", regex=False)
                .str.replace(" ", "", regex=False)
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
            )
            
            df_copy[col] = pd.to_numeric(df_copy[col], errors="coerce")
    
    return df_copy


print("Función force_numeric_columns creada correctamente.")

In [ ]:
# ============================================================
# 5. FUNCIÓN DE EXPORTACIÓN: CSV NORMAL + CSV EXCEL ES + XLSX
# ============================================================

def export_table_files(dataframe, output_folder, file_name, sheet_name="Data"):
    """
    Exporta un DataFrame en tres formatos:
    
    1. CSV estándar para Python.
    2. CSV compatible con Excel España: separador ; y decimal ,
    3. XLSX con formato numérico.
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    csv_path = f"{output_folder}/{file_name}.csv"
    csv_excel_es_path = f"{output_folder}/{file_name}_excel_es.csv"
    xlsx_path = f"{output_folder}/{file_name}.xlsx"
    
    df_export = dataframe.copy()
    
    for col in df_export.columns:
        if pd.api.types.is_datetime64_any_dtype(df_export[col]):
            df_export[col] = df_export[col].dt.strftime("%d/%m/%Y %H:%M")
    
    # CSV estándar
    dataframe.to_csv(csv_path, index=False, encoding="utf-8-sig")
    
    # CSV compatible con Excel España
    df_export.to_csv(
        csv_excel_es_path,
        index=False,
        sep=";",
        decimal=",",
        encoding="utf-8-sig"
    )
    
    # Excel XLSX
    safe_sheet_name = sheet_name[:31]
    
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        dataframe.to_excel(writer, index=False, sheet_name=safe_sheet_name)
        
        worksheet = writer.sheets[safe_sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        
        for cell in worksheet[1]:
            cell.font = Font(bold=True)
        
        for col_idx, col_name in enumerate(dataframe.columns, start=1):
            column_letter = get_column_letter(col_idx)
            
            if pd.api.types.is_float_dtype(dataframe[col_name]):
                if col_name in [
                    "occupancy_rate",
                    "avg_occupancy",
                    "margin_pct",
                    "avg_margin_pct",
                    "price_gap_pct",
                    "revenue_uplift_pct",
                    "margin_uplift_pct",
                    "price_change_pct",
                    "demand_change_pct",
                    "best_revenue_price_change_pct",
                    "best_margin_price_change_pct",
                    "best_revenue_uplift_pct",
                    "best_margin_uplift_pct"
                ]:
                    number_format = "0.00%"
                else:
                    number_format = "0.00"
                
                for cell in worksheet[column_letter][1:]:
                    cell.number_format = number_format
            
            elif pd.api.types.is_integer_dtype(dataframe[col_name]):
                for cell in worksheet[column_letter][1:]:
                    cell.number_format = "0"
            
            elif pd.api.types.is_datetime64_any_dtype(dataframe[col_name]):
                for cell in worksheet[column_letter][1:]:
                    cell.number_format = "DD/MM/YYYY HH:MM"
        
        for column_cells in worksheet.columns:
            max_length = 0
            column_letter = get_column_letter(column_cells[0].column)
            
            for cell in column_cells:
                if cell.value is not None:
                    max_length = max(max_length, len(str(cell.value)))
            
            adjusted_width = min(max_length + 2, 40)
            worksheet.column_dimensions[column_letter].width = adjusted_width
    
    print(f"Exportado: {file_name}")


print("Función export_table_files creada correctamente.")

In [ ]:
# ============================================================
# 6. FUNCIONES DE CLASIFICACIÓN DE ELASTICIDAD
# ============================================================

def classify_elasticity(elasticity):
    """
    Clasifica la elasticidad precio-demanda.
    """
    
    if pd.isna(elasticity):
        return "Not enough data"
    
    if elasticity >= 0:
        return "Positive / Confounded"
    elif elasticity > -0.5:
        return "Very inelastic"
    elif elasticity > -1:
        return "Inelastic"
    elif elasticity >= -1.2:
        return "Unit elastic area"
    elif elasticity >= -2:
        return "Elastic"
    else:
        return "Very elastic"


def elasticity_business_interpretation(elasticity):
    """
    Traduce elasticidad a interpretación de negocio.
    """
    
    if pd.isna(elasticity):
        return "Insufficient information to estimate price response."
    
    if elasticity >= 0:
        return "The relationship appears positive, probably affected by seasonality or demand pressure. Review with controls."
    elif elasticity > -1:
        return "Demand is relatively inelastic. Controlled price increases may be tested if occupancy and margin are healthy."
    elif elasticity >= -1.2:
        return "Demand is close to unit elasticity. Price changes should be tested carefully."
    else:
        return "Demand is elastic. Price increases may significantly reduce demand; promotions may be more effective in low occupancy contexts."


def recommendation_from_elasticity(row):
    """
    Recomendación ejecutiva combinando elasticidad, ocupación, margen y posición competitiva.
    """
    
    elasticity = row["elasticity_for_simulation"]
    occupancy = row["avg_occupancy"]
    margin_pct = row["avg_margin_pct"]
    price_index = row["avg_price_index"]
    
    if occupancy >= 0.85 and elasticity > -1 and margin_pct >= 0.25 and price_index <= 1.08:
        return "Test controlled price increase"
    elif occupancy >= 0.85 and price_index > 1.08:
        return "Maintain price; already premium positioned"
    elif occupancy >= 0.80 and margin_pct < 0.20:
        return "Review price-cost balance"
    elif occupancy < 0.55 and elasticity <= -1:
        return "Test promotional scenario"
    elif occupancy < 0.55 and margin_pct >= 0.25:
        return "Improve demand generation"
    else:
        return "Maintain and monitor"


print("Funciones de clasificación creadas correctamente.")

## 3. Carga y revisión del dataset

Leemos el dataset generado en el Notebook 1.

Cada fila representa un viaje de ferry con información de precio, demanda, ocupación, revenue, costes y margen.

In [ ]:
# ============================================================
# 7. CARGA DEL DATASET
# ============================================================

df = pd.read_csv(data_path, parse_dates=["trip_date", "departure_datetime"])

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

df.head()

In [ ]:
# ============================================================
# 8. ASEGURAR TIPOS NUMÉRICOS
# ============================================================

numeric_columns_to_check = [
    "capacity",
    "base_price",
    "avg_ticket_price",
    "competitor_price",
    "price_index_vs_competitor",
    "route_elasticity",
    "days_before_departure",
    "weather_score",
    "event_flag",
    "expected_demand",
    "tickets_sold",
    "occupancy_rate",
    "revenue",
    "fixed_operational_cost",
    "variable_cost_per_passenger",
    "total_operational_cost",
    "margin",
    "margin_pct"
]

df = force_numeric_columns(df, numeric_columns_to_check)

print("Columnas numéricas revisadas correctamente.")
df[numeric_columns_to_check].dtypes

In [ ]:
# ============================================================
# 9. VALIDACIONES BÁSICAS
# ============================================================

print("Validación 1: tickets_sold nunca debe superar capacity")
print((df["tickets_sold"] <= df["capacity"]).all())

print("\nValidación 2: occupancy_rate entre 0 y 1")
print(df["occupancy_rate"].between(0, 1).all())

print("\nValidación 3: avg_ticket_price positivo")
print((df["avg_ticket_price"] > 0).all())

print("\nValidación 4: tickets_sold positivo o cero")
print((df["tickets_sold"] >= 0).all())

print("\nNulos totales:")
print(df.isnull().sum().sum())

## 4. KPIs generales

Antes de estimar elasticidad, revisamos la foto general del dataset.

In [ ]:
# ============================================================
# 10. KPIS GENERALES
# ============================================================

total_trips = df["trip_id"].nunique()
total_tickets = df["tickets_sold"].sum()
total_revenue = df["revenue"].sum()
total_margin = df["margin"].sum()
avg_occupancy = df["occupancy_rate"].mean()
avg_ticket_price = df["avg_ticket_price"].mean()
avg_margin_pct = df["margin_pct"].mean()

print("KPIs GENERALES")
print("-" * 45)
print(f"Número total de viajes: {total_trips:,.0f}")
print(f"Tickets vendidos: {total_tickets:,.0f}")
print(f"Revenue total: {total_revenue:,.2f} €")
print(f"Margen total: {total_margin:,.2f} €")
print(f"Ocupación media: {avg_occupancy:.2%}")
print(f"Precio medio del ticket: {avg_ticket_price:.2f} €")
print(f"Margen porcentual medio: {avg_margin_pct:.2%}")

## 5. Concepto de elasticidad precio-demanda

La elasticidad precio-demanda se puede interpretar así:

$$Elasticidad = \frac{\%\ cambio\ en\ demanda}{\%\ cambio\ en\ precio}$$

Ejemplo sencillo:

- Si el precio sube un 10%.
- Y la demanda cae un 5%.
- La elasticidad sería -0,5.

Interpretación:

- Entre 0 y -1: demanda inelástica.
- Cerca de -1: elasticidad unitaria.
- Menor que -1: demanda elástica.

En este notebook usaremos modelos log-log porque el coeficiente de `log(precio)` se interpreta directamente como elasticidad.

## 6. Preparación de variables para elasticidad

Para estimar elasticidad necesitamos valores positivos de precio y demanda, porque vamos a usar logaritmos.

Usaremos `tickets_sold` como proxy de demanda observada.

Nota de negocio: en la realidad, si un ferry alcanza capacidad máxima, la demanda observada queda censurada. Podría haber más demanda no capturada. Por eso interpretaremos los resultados con cuidado.

In [ ]:
# ============================================================
# 11. PREPARACIÓN DE DATASET PARA ELASTICIDAD
# ============================================================

elasticity_df = df[
    (df["avg_ticket_price"] > 0) &
    (df["tickets_sold"] > 0)
].copy()

elasticity_df["log_price"] = np.log(elasticity_df["avg_ticket_price"])
elasticity_df["log_demand"] = np.log(elasticity_df["tickets_sold"])

print(f"Filas disponibles para análisis de elasticidad: {len(elasticity_df):,}")
elasticity_df[[
    "route",
    "season",
    "avg_ticket_price",
    "tickets_sold",
    "log_price",
    "log_demand"
]].head()

## 7. Elasticidad simple por ruta

Primero calculamos una elasticidad básica por ruta mediante una regresión log-log simple:

`log(demanda) = a + b * log(precio)`

El coeficiente `b` es la elasticidad estimada.

Importante: esta versión es muy útil como primera aproximación, pero puede estar afectada por temporada, eventos, clima, fin de semana o antelación de compra.

In [ ]:
# ============================================================
# 12. ELASTICIDAD SIMPLE POR RUTA
# ============================================================

simple_elasticity_results = []

for route, group in elasticity_df.groupby("route"):
    
    if len(group) < 30:
        continue
    
    X = group[["log_price"]].values
    y = group["log_demand"].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    y_pred = model.predict(X)
    elasticity = model.coef_[0]
    r2 = r2_score(y, y_pred)
    
    price_demand_corr = group["avg_ticket_price"].corr(group["tickets_sold"])
    
    simple_elasticity_results.append({
        "route": route,
        "observations": len(group),
        "simple_elasticity": elasticity,
        "simple_r2": r2,
        "price_demand_corr": price_demand_corr,
        "avg_ticket_price": group["avg_ticket_price"].mean(),
        "avg_tickets_sold": group["tickets_sold"].mean(),
        "avg_occupancy": group["occupancy_rate"].mean(),
        "total_revenue": group["revenue"].sum(),
        "avg_margin_pct": group["margin_pct"].mean(),
        "simulated_route_elasticity": group["route_elasticity"].mean()
    })

simple_elasticity = pd.DataFrame(simple_elasticity_results)

simple_elasticity["elasticity_class"] = simple_elasticity["simple_elasticity"].apply(classify_elasticity)
simple_elasticity["business_interpretation"] = simple_elasticity["simple_elasticity"].apply(elasticity_business_interpretation)

simple_elasticity = simple_elasticity.sort_values("simple_elasticity")

simple_elasticity

### Lectura de negocio

La elasticidad simple nos da una primera señal, pero no debe usarse sola para tomar decisiones.

Puede aparecer una relación positiva entre precio y demanda porque en temporada alta suben las dos cosas a la vez:

- Sube la demanda.
- Sube el precio.
- Parece que mayor precio genera más demanda, pero en realidad el factor oculto es la temporada.

Por eso ahora calcularemos una elasticidad ajustada.

## 8. Elasticidad ajustada por ruta

Ahora estimamos elasticidad controlando otras variables que también influyen en la demanda:

- Temporada.
- Día de la semana.
- Hora de salida.
- Fin de semana.
- Evento.
- Clima.
- Antelación de compra.
- Ventana de reserva.
- Posición frente al competidor.

La fórmula conceptual sería:

`log(demanda) = precio + temporada + día + hora + evento + clima + antelación + competidor`

El coeficiente de `log_price` se interpreta como elasticidad ajustada.

In [ ]:
# ============================================================
# 13. FUNCIÓN PARA ESTIMAR ELASTICIDAD AJUSTADA POR RUTA
# ============================================================

def estimate_adjusted_elasticity_by_route(dataframe):
    """
    Estima elasticidad ajustada por ruta mediante regresión lineal log-log
    con variables de control.
    """
    
    results = []
    
    control_columns = [
        "log_price",
        "weather_score",
        "days_before_departure",
        "event_flag",
        "is_weekend",
        "high_season_flag",
        "price_index_vs_competitor",
        "season",
        "day_of_week",
        "booking_window",
        "departure_hour"
    ]
    
    for route, group in dataframe.groupby("route"):
        
        route_data = group[control_columns + ["log_demand"]].dropna().copy()
        
        if len(route_data) < 50:
            continue
        
        X = route_data.drop(columns=["log_demand"])
        y = route_data["log_demand"]
        
        X_encoded = pd.get_dummies(
            X,
            columns=["season", "day_of_week", "booking_window", "departure_hour"],
            drop_first=True
        )
        
        model = LinearRegression()
        model.fit(X_encoded, y)
        
        y_pred = model.predict(X_encoded)
        r2 = r2_score(y, y_pred)
        
        coefficients = pd.Series(model.coef_, index=X_encoded.columns)
        
        adjusted_elasticity = coefficients.get("log_price", np.nan)
        
        results.append({
            "route": route,
            "observations": len(route_data),
            "adjusted_elasticity": adjusted_elasticity,
            "adjusted_r2": r2,
            "avg_price_index": group["price_index_vs_competitor"].mean(),
            "avg_ticket_price": group["avg_ticket_price"].mean(),
            "avg_occupancy": group["occupancy_rate"].mean(),
            "total_revenue": group["revenue"].sum(),
            "total_margin": group["margin"].sum(),
            "avg_margin_pct": group["margin_pct"].mean(),
            "simulated_route_elasticity": group["route_elasticity"].mean()
        })
    
    results_df = pd.DataFrame(results)
    
    results_df["elasticity_class"] = results_df["adjusted_elasticity"].apply(classify_elasticity)
    results_df["business_interpretation"] = results_df["adjusted_elasticity"].apply(elasticity_business_interpretation)
    
    return results_df.sort_values("adjusted_elasticity")


adjusted_elasticity = estimate_adjusted_elasticity_by_route(elasticity_df)

adjusted_elasticity

## 9. Comparación de elasticidades

Comparamos tres referencias:

1. Elasticidad simple estimada.
2. Elasticidad ajustada estimada.
3. Elasticidad simulada incorporada en el dataset.

En un caso real no tendríamos la elasticidad simulada, pero aquí nos sirve como referencia porque estamos trabajando con datos sintéticos.

In [ ]:
# ============================================================
# 14. COMPARACIÓN DE ELASTICIDADES
# ============================================================

elasticity_comparison = (
    simple_elasticity[[
        "route",
        "simple_elasticity",
        "simple_r2",
        "price_demand_corr"
    ]]
    .merge(
        adjusted_elasticity[[
            "route",
            "adjusted_elasticity",
            "adjusted_r2",
            "simulated_route_elasticity",
            "avg_price_index",
            "avg_ticket_price",
            "avg_occupancy",
            "total_revenue",
            "total_margin",
            "avg_margin_pct"
        ]],
        on="route",
        how="left"
    )
)

# Elasticidad que usaremos para simulación:
# Si la elasticidad ajustada es negativa y razonable, usamos esa.
# Si aparece positiva o extrema por ruido/confusión, usamos la elasticidad de negocio simulada como prior.
elasticity_comparison["elasticity_for_simulation"] = np.where(
    (elasticity_comparison["adjusted_elasticity"] < -0.05) &
    (elasticity_comparison["adjusted_elasticity"] > -2.50),
    elasticity_comparison["adjusted_elasticity"],
    elasticity_comparison["simulated_route_elasticity"]
)

elasticity_comparison["elasticity_class"] = elasticity_comparison["elasticity_for_simulation"].apply(classify_elasticity)
elasticity_comparison["business_interpretation"] = elasticity_comparison["elasticity_for_simulation"].apply(elasticity_business_interpretation)

elasticity_comparison = elasticity_comparison.sort_values("elasticity_for_simulation")

elasticity_comparison

### Lectura de negocio

La columna más importante para los siguientes pasos será:

`elasticity_for_simulation`

Esta columna representa la elasticidad que usaremos para simular escenarios.

Criterio usado:

- Si la elasticidad ajustada es negativa y razonable, se usa la estimación ajustada.
- Si sale positiva o poco fiable, se usa la elasticidad simulada del dataset como prior de negocio.

Esto evita tomar decisiones absurdas por una estimación estadística contaminada por estacionalidad, capacidad o ruido.

## 10. Visualización de elasticidad por ruta

Una línea vertical en `-1` nos ayuda a separar rutas inelásticas de rutas elásticas.

In [ ]:
# ============================================================
# 15. GRÁFICO: ELASTICIDAD POR RUTA
# ============================================================

plt.figure(figsize=(12, 6))

plot_df = elasticity_comparison.sort_values("elasticity_for_simulation")

sns.barplot(
    data=plot_df,
    x="elasticity_for_simulation",
    y="route"
)

plt.axvline(-1, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)

plt.title("Elasticidad precio-demanda estimada por ruta")
plt.xlabel("Elasticidad")
plt.ylabel("Ruta")
plt.tight_layout()

plt.savefig(f"{images_path}/21_elasticity_by_route.png", dpi=300, bbox_inches="tight")

plt.show()

## 11. Análisis visual: precio vs demanda por ruta

Este gráfico ayuda a visualizar la relación entre precio y tickets vendidos.

Recordatorio: no debemos interpretar cada punto de forma aislada, porque hay factores como temporada, eventos, día de la semana y capacidad.

In [ ]:
# ============================================================
# 16. SCATTER PRECIO VS DEMANDA POR RUTA
# ============================================================

plt.figure(figsize=(12, 7))

sample_df = elasticity_df.sample(min(2500, len(elasticity_df)), random_state=42)

sns.scatterplot(
    data=sample_df,
    x="avg_ticket_price",
    y="tickets_sold",
    hue="route",
    alpha=0.55
)

plt.title("Relación entre precio medio y tickets vendidos")
plt.xlabel("Precio medio ticket (€)")
plt.ylabel("Tickets vendidos")
plt.tight_layout()

plt.savefig(f"{images_path}/22_price_vs_demand_by_route.png", dpi=300, bbox_inches="tight")

plt.show()

## 12. Mapa de diagnóstico: ocupación vs elasticidad

Este gráfico es especialmente útil para negocio:

- Alta ocupación + baja elasticidad: posible subida controlada.
- Baja ocupación + alta elasticidad: posible promoción.
- Alta ocupación + alta elasticidad: cuidado con subir precio.
- Baja ocupación + baja elasticidad: revisar propuesta de valor o generación de demanda.

In [ ]:
# ============================================================
# 17. DIAGNÓSTICO: OCUPACIÓN VS ELASTICIDAD
# ============================================================

diagnosis_plot = elasticity_comparison.copy()

diagnosis_plot["route_recommendation"] = diagnosis_plot.apply(recommendation_from_elasticity, axis=1)

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=diagnosis_plot,
    x="elasticity_for_simulation",
    y="avg_occupancy",
    size="total_revenue",
    hue="route_recommendation",
    sizes=(100, 900),
    alpha=0.75
)

plt.axvline(-1, linestyle="--", linewidth=1)
plt.axhline(0.85, linestyle="--", linewidth=1)
plt.axhline(0.55, linestyle="--", linewidth=1)

plt.title("Mapa de diagnóstico: elasticidad vs ocupación")
plt.xlabel("Elasticidad para simulación")
plt.ylabel("Ocupación media")
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.savefig(f"{images_path}/23_elasticity_vs_occupancy_diagnosis.png", dpi=300, bbox_inches="tight")

plt.show()

## 13. Elasticidad por ruta y temporada

Ahora analizamos si la sensibilidad al precio cambia por temporada.

Esto es importante porque una misma ruta puede comportarse de forma distinta en temporada baja, media o alta.

In [ ]:
# ============================================================
# 18. ELASTICIDAD SIMPLE POR RUTA Y TEMPORADA
# ============================================================

route_season_elasticity_results = []

for (route, season), group in elasticity_df.groupby(["route", "season"]):
    
    if len(group) < 30:
        continue
    
    X = group[["log_price"]].values
    y = group["log_demand"].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    y_pred = model.predict(X)
    elasticity = model.coef_[0]
    r2 = r2_score(y, y_pred)
    
    route_season_elasticity_results.append({
        "route": route,
        "season": season,
        "observations": len(group),
        "route_season_elasticity": elasticity,
        "r2": r2,
        "avg_ticket_price": group["avg_ticket_price"].mean(),
        "avg_occupancy": group["occupancy_rate"].mean(),
        "total_revenue": group["revenue"].sum(),
        "avg_margin_pct": group["margin_pct"].mean()
    })

route_season_elasticity = pd.DataFrame(route_season_elasticity_results)

season_order = ["Low", "Shoulder", "High"]

route_season_elasticity["season"] = pd.Categorical(
    route_season_elasticity["season"],
    categories=season_order,
    ordered=True
)

route_season_elasticity["elasticity_class"] = route_season_elasticity["route_season_elasticity"].apply(classify_elasticity)

route_season_elasticity = route_season_elasticity.sort_values(["route", "season"])

route_season_elasticity

In [ ]:
# ============================================================
# 19. HEATMAP: ELASTICIDAD POR RUTA Y TEMPORADA
# ============================================================

elasticity_pivot = route_season_elasticity.pivot(
    index="route",
    columns="season",
    values="route_season_elasticity"
)

elasticity_pivot = elasticity_pivot[season_order]

plt.figure(figsize=(10, 6))

sns.heatmap(
    elasticity_pivot,
    annot=True,
    fmt=".2f",
    center=-1,
    cmap="coolwarm"
)

plt.title("Elasticidad precio-demanda por ruta y temporada")
plt.xlabel("Temporada")
plt.ylabel("Ruta")
plt.tight_layout()

plt.savefig(f"{images_path}/24_heatmap_route_season_elasticity.png", dpi=300, bbox_inches="tight")

plt.show()

## 14. Simulación de escenarios de precio

Ahora simulamos distintos cambios de precio y estimamos su impacto sobre:

- Tickets vendidos.
- Ocupación.
- Revenue.
- Coste operativo.
- Margen.
- Margen porcentual.

Escenarios:

- -15%
- -10%
- -5%
- 0%
- +5%
- +10%
- +15%

Usaremos un modelo de elasticidad constante:

`demanda nueva = demanda actual * (precio nuevo / precio actual) ^ elasticidad`

In [ ]:
# ============================================================
# 20. PREPARAR MAPA DE ELASTICIDAD PARA SIMULACIÓN
# ============================================================

elasticity_map = dict(
    zip(
        elasticity_comparison["route"],
        elasticity_comparison["elasticity_for_simulation"]
    )
)

df_sim = df.copy()
df_sim["elasticity_for_simulation"] = df_sim["route"].map(elasticity_map)

# Fallback por seguridad
df_sim["elasticity_for_simulation"] = df_sim["elasticity_for_simulation"].fillna(df_sim["route_elasticity"])

df_sim[[
    "route",
    "avg_ticket_price",
    "tickets_sold",
    "capacity",
    "revenue",
    "margin",
    "elasticity_for_simulation"
]].head()

In [ ]:
# ============================================================
# 21. SIMULACIÓN DE ESCENARIOS DE PRECIO
# ============================================================

price_changes = [-0.15, -0.10, -0.05, 0.00, 0.05, 0.10, 0.15]

simulation_rows = []

for price_change in price_changes:
    
    scenario = df_sim.copy()
    
    scenario["price_change_pct"] = price_change
    scenario["scenario_label"] = f"{price_change:+.0%}"
    
    scenario["simulated_ticket_price"] = scenario["avg_ticket_price"] * (1 + price_change)
    
    scenario["simulated_demand"] = (
        scenario["tickets_sold"] *
        ((scenario["simulated_ticket_price"] / scenario["avg_ticket_price"]) ** scenario["elasticity_for_simulation"])
    )
    
    scenario["simulated_tickets_sold"] = np.minimum(
        scenario["simulated_demand"],
        scenario["capacity"]
    )
    
    scenario["simulated_occupancy_rate"] = scenario["simulated_tickets_sold"] / scenario["capacity"]
    
    scenario["simulated_revenue"] = (
        scenario["simulated_tickets_sold"] *
        scenario["simulated_ticket_price"]
    )
    
    scenario["simulated_operational_cost"] = (
        scenario["fixed_operational_cost"] +
        scenario["simulated_tickets_sold"] * scenario["variable_cost_per_passenger"]
    )
    
    scenario["simulated_margin"] = (
        scenario["simulated_revenue"] -
        scenario["simulated_operational_cost"]
    )
    
    scenario["simulated_margin_pct"] = np.where(
        scenario["simulated_revenue"] > 0,
        scenario["simulated_margin"] / scenario["simulated_revenue"],
        0
    )
    
    route_scenario = (
        scenario
        .groupby(["route", "scenario_label", "price_change_pct"])
        .agg(
            trips=("trip_id", "count"),
            current_tickets=("tickets_sold", "sum"),
            simulated_tickets=("simulated_tickets_sold", "sum"),
            current_revenue=("revenue", "sum"),
            simulated_revenue=("simulated_revenue", "sum"),
            current_margin=("margin", "sum"),
            simulated_margin=("simulated_margin", "sum"),
            current_avg_occupancy=("occupancy_rate", "mean"),
            simulated_avg_occupancy=("simulated_occupancy_rate", "mean"),
            current_avg_price=("avg_ticket_price", "mean"),
            simulated_avg_price=("simulated_ticket_price", "mean"),
            simulated_margin_pct=("simulated_margin_pct", "mean")
        )
        .reset_index()
    )
    
    simulation_rows.append(route_scenario)

scenario_simulation = pd.concat(simulation_rows, ignore_index=True)

scenario_simulation["revenue_uplift"] = (
    scenario_simulation["simulated_revenue"] -
    scenario_simulation["current_revenue"]
)

scenario_simulation["revenue_uplift_pct"] = np.where(
    scenario_simulation["current_revenue"] > 0,
    scenario_simulation["revenue_uplift"] / scenario_simulation["current_revenue"],
    0
)

scenario_simulation["margin_uplift"] = (
    scenario_simulation["simulated_margin"] -
    scenario_simulation["current_margin"]
)

scenario_simulation["margin_uplift_pct"] = np.where(
    scenario_simulation["current_margin"] != 0,
    scenario_simulation["margin_uplift"] / scenario_simulation["current_margin"],
    0
)

scenario_simulation.head(10)

## 15. Mejor escenario por ruta

Calculamos qué escenario maximiza revenue y qué escenario maximiza margen.

En negocio, no siempre elegiremos el escenario de máximo revenue. A veces interesa proteger margen, ocupación o experiencia cliente.

In [ ]:
# ============================================================
# 22. MEJORES ESCENARIOS POR RUTA
# ============================================================

best_revenue_scenario = (
    scenario_simulation
    .sort_values(["route", "simulated_revenue"], ascending=[True, False])
    .groupby("route")
    .head(1)
    .copy()
)

best_revenue_scenario["optimization_target"] = "Max revenue"

best_margin_scenario = (
    scenario_simulation
    .sort_values(["route", "simulated_margin"], ascending=[True, False])
    .groupby("route")
    .head(1)
    .copy()
)

best_margin_scenario["optimization_target"] = "Max margin"

best_scenarios = pd.concat(
    [best_revenue_scenario, best_margin_scenario],
    ignore_index=True
)

best_scenarios = best_scenarios.sort_values(["route", "optimization_target"])

best_scenarios

## 16. Visualización de escenarios por ruta

Este gráfico muestra cómo cambia el revenue estimado según el cambio de precio aplicado.

In [ ]:
# ============================================================
# 23. GRÁFICO: ESCENARIOS DE REVENUE POR RUTA
# ============================================================

plt.figure(figsize=(12, 7))

sns.lineplot(
    data=scenario_simulation,
    x="price_change_pct",
    y="simulated_revenue",
    hue="route",
    marker="o"
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.title("Simulación de revenue según cambio de precio")
plt.xlabel("Cambio de precio")
plt.ylabel("Revenue simulado (€)")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()

plt.savefig(f"{images_path}/25_revenue_scenario_simulation_by_route.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# 24. HEATMAP: IMPACTO EN REVENUE POR ESCENARIO
# ============================================================

revenue_uplift_pivot = scenario_simulation.pivot(
    index="route",
    columns="scenario_label",
    values="revenue_uplift_pct"
)

scenario_order = ["-15%", "-10%", "-5%", "+0%", "+5%", "+10%", "+15%"]
revenue_uplift_pivot = revenue_uplift_pivot[scenario_order]

plt.figure(figsize=(12, 6))

sns.heatmap(
    revenue_uplift_pivot,
    annot=True,
    fmt=".2%",
    center=0,
    cmap="RdYlGn"
)

plt.title("Impacto estimado en revenue por escenario de precio")
plt.xlabel("Cambio de precio")
plt.ylabel("Ruta")
plt.tight_layout()

plt.savefig(f"{images_path}/26_heatmap_revenue_uplift_scenarios.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# 25. HEATMAP: IMPACTO EN MARGEN POR ESCENARIO
# ============================================================

margin_uplift_pivot = scenario_simulation.pivot(
    index="route",
    columns="scenario_label",
    values="margin_uplift_pct"
)

margin_uplift_pivot = margin_uplift_pivot[scenario_order]

plt.figure(figsize=(12, 6))

sns.heatmap(
    margin_uplift_pivot,
    annot=True,
    fmt=".2%",
    center=0,
    cmap="RdYlGn"
)

plt.title("Impacto estimado en margen por escenario de precio")
plt.xlabel("Cambio de precio")
plt.ylabel("Ruta")
plt.tight_layout()

plt.savefig(f"{images_path}/27_heatmap_margin_uplift_scenarios.png", dpi=300, bbox_inches="tight")

plt.show()

## 17. Diagnóstico final por ruta

Combinamos:

- Elasticidad.
- Ocupación.
- Margen.
- Posición frente a competidor.
- Mejor escenario de revenue.
- Mejor escenario de margen.

Esto nos permite generar recomendaciones ejecutivas.

In [ ]:
# ============================================================
# 26. DIAGNÓSTICO FINAL POR RUTA
# ============================================================

route_diagnosis = elasticity_comparison.copy()

route_diagnosis["recommendation"] = route_diagnosis.apply(recommendation_from_elasticity, axis=1)

best_revenue_compact = best_revenue_scenario[[
    "route",
    "scenario_label",
    "price_change_pct",
    "simulated_revenue",
    "revenue_uplift",
    "revenue_uplift_pct"
]].rename(columns={
    "scenario_label": "best_revenue_scenario",
    "price_change_pct": "best_revenue_price_change_pct",
    "simulated_revenue": "best_revenue_simulated_revenue",
    "revenue_uplift": "best_revenue_uplift",
    "revenue_uplift_pct": "best_revenue_uplift_pct"
})

best_margin_compact = best_margin_scenario[[
    "route",
    "scenario_label",
    "price_change_pct",
    "simulated_margin",
    "margin_uplift",
    "margin_uplift_pct"
]].rename(columns={
    "scenario_label": "best_margin_scenario",
    "price_change_pct": "best_margin_price_change_pct",
    "simulated_margin": "best_margin_simulated_margin",
    "margin_uplift": "best_margin_uplift",
    "margin_uplift_pct": "best_margin_uplift_pct"
})

route_diagnosis = (
    route_diagnosis
    .merge(best_revenue_compact, on="route", how="left")
    .merge(best_margin_compact, on="route", how="left")
)

route_diagnosis = route_diagnosis[[
    "route",
    "avg_ticket_price",
    "avg_occupancy",
    "avg_margin_pct",
    "avg_price_index",
    "simple_elasticity",
    "adjusted_elasticity",
    "simulated_route_elasticity",
    "elasticity_for_simulation",
    "elasticity_class",
    "total_revenue",
    "total_margin",
    "best_revenue_scenario",
    "best_revenue_uplift_pct",
    "best_margin_scenario",
    "best_margin_uplift_pct",
    "recommendation",
    "business_interpretation"
]].sort_values("total_revenue", ascending=False)

route_diagnosis

## Simulador de punto de equilibrio para decisiones de pricing

Este bloque añade una herramienta interactiva al Notebook 3.

La finalidad es complementar el análisis de elasticidad con una pregunta muy práctica:

> ¿A partir de qué ocupación, precio o volumen empieza a ser rentable una salida?

El simulador combina:

- Precio actual.
- Cambio de precio.
- Elasticidad precio-demanda.
- Demanda externa.
- Capacidad.
- Coste fijo.
- Coste variable.
- Punto de equilibrio.
- Margen simulado.

Esto encaja especialmente bien en el Notebook 3 porque conecta directamente elasticidad con decisión de pricing.

In [ ]:
# ============================================================
# BREAK-EVEN TOOL 1. PREPARAR DATOS DEL SIMULADOR
# ============================================================

break_even_base = (
    df
    .groupby("route")
    .agg(
        avg_ticket_price=("avg_ticket_price", "mean"),
        avg_occupancy=("occupancy_rate", "mean"),
        avg_capacity=("capacity", "mean"),
        avg_fixed_cost=("fixed_operational_cost", "mean"),
        avg_variable_cost=("variable_cost_per_passenger", "mean"),
        avg_competitor_price=("competitor_price", "mean"),
        current_revenue=("revenue", "sum"),
        current_margin=("margin", "sum"),
        trips=("trip_id", "count")
    )
    .reset_index()
)

break_even_base = break_even_base.merge(
    elasticity_comparison[["route", "elasticity_for_simulation"]],
    on="route",
    how="left"
)

break_even_base["elasticity_for_simulation"] = break_even_base["elasticity_for_simulation"].fillna(
    df.groupby("route")["route_elasticity"].mean().reindex(break_even_base["route"]).values
)

break_even_base["current_tickets_per_trip"] = (
    break_even_base["avg_capacity"] * break_even_base["avg_occupancy"]
)

break_even_base["current_revenue_per_trip"] = (
    break_even_base["current_revenue"] / break_even_base["trips"]
)

break_even_base["current_margin_per_trip"] = (
    break_even_base["current_margin"] / break_even_base["trips"]
)

break_even_base

In [ ]:
# ============================================================
# BREAK-EVEN TOOL 2. SIMULADOR INTERACTIVO EN COLAB
# ============================================================

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output

    route_widget = widgets.Dropdown(
        options=sorted(break_even_base["route"].unique()),
        description="Ruta:",
        layout=widgets.Layout(width="420px")
    )

    price_change_widget = widgets.IntSlider(
        value=0,
        min=-25,
        max=30,
        step=1,
        description="Precio:",
        continuous_update=False,
        layout=widgets.Layout(width="540px")
    )

    demand_factor_widget = widgets.IntSlider(
        value=100,
        min=70,
        max=135,
        step=1,
        description="Demanda:",
        continuous_update=False,
        layout=widgets.Layout(width="540px")
    )

    fixed_cost_widget = widgets.IntSlider(
        value=0,
        min=-20,
        max=30,
        step=1,
        description="Coste fijo:",
        continuous_update=False,
        layout=widgets.Layout(width="540px")
    )

    variable_cost_widget = widgets.IntSlider(
        value=0,
        min=-20,
        max=30,
        step=1,
        description="Coste var.:",
        continuous_update=False,
        layout=widgets.Layout(width="540px")
    )

    be_output = widgets.Output()

    def eur(value):
        return f"{value:,.0f} €".replace(",", "X").replace(".", ",").replace("X", ".")

    def eur2(value):
        return f"{value:,.2f} €".replace(",", "X").replace(".", ",").replace("X", ".")

    def pct_es(value):
        return f"{value:.1%}".replace(".", ",")

    def render_break_even_tool(*args):
        with be_output:
            clear_output(wait=True)

            route = route_widget.value
            price_change = price_change_widget.value / 100
            external_demand = demand_factor_widget.value / 100
            fixed_cost_shock = fixed_cost_widget.value / 100
            variable_cost_shock = variable_cost_widget.value / 100

            row = break_even_base[break_even_base["route"] == route].iloc[0]

            base_price = row["avg_ticket_price"]
            simulated_price = base_price * (1 + price_change)

            capacity = row["avg_capacity"]
            elasticity = row["elasticity_for_simulation"]
            current_tickets = row["current_tickets_per_trip"]

            fixed_cost = row["avg_fixed_cost"] * (1 + fixed_cost_shock)
            variable_cost = row["avg_variable_cost"] * (1 + variable_cost_shock)

            simulated_demand = (
                current_tickets *
                external_demand *
                ((simulated_price / base_price) ** elasticity)
            )

            simulated_tickets = min(max(simulated_demand, 0), capacity)
            simulated_occupancy = simulated_tickets / capacity

            revenue = simulated_tickets * simulated_price
            total_cost = fixed_cost + simulated_tickets * variable_cost
            margin = revenue - total_cost
            margin_pct = margin / revenue if revenue > 0 else 0

            contribution_margin = simulated_price - variable_cost
            break_even_tickets = fixed_cost / contribution_margin if contribution_margin > 0 else np.inf
            break_even_occupancy = break_even_tickets / capacity if contribution_margin > 0 else np.inf
            break_even_price = variable_cost + fixed_cost / simulated_tickets if simulated_tickets > 0 else np.inf

            current_revenue = row["current_revenue_per_trip"]
            current_margin = row["current_margin_per_trip"]
            revenue_delta = revenue / current_revenue - 1 if current_revenue != 0 else 0
            margin_delta = margin / current_margin - 1 if current_margin != 0 else 0

            price_index = simulated_price / row["avg_competitor_price"]

            if margin >= 0 and simulated_occupancy >= break_even_occupancy:
                status = "Equilibrio superado"
                status_color = "#147d64"
            elif margin >= 0:
                status = "Rentable, pero revisar ocupación"
                status_color = "#b7791f"
            else:
                status = "No alcanza equilibrio"
                status_color = "#b5473a"

            if margin >= 0 and revenue_delta > 0 and margin_delta > 0 and price_index <= 1.12:
                decision = "Escenario atractivo: mejora revenue y margen sin quedar excesivamente premium."
            elif margin >= 0 and margin_delta > 0:
                decision = "Escenario orientado a margen: revisar posible impacto comercial."
            elif margin >= 0 and simulated_occupancy < 0.55:
                decision = "Rentable pero frágil: la ocupación queda baja."
            else:
                decision = "Escenario débil: revisar precio, costes o generación de demanda."

            cards_html = f'''
            <div style="font-family:Arial; margin-top:10px;">
                <h3 style="margin-bottom:6px;">Punto de equilibrio — {route}</h3>
                <p style="margin-top:0; color:#666;">Estado: <b style="color:{status_color};">{status}</b></p>
                <div style="display:grid; grid-template-columns: repeat(4, 1fr); gap:10px;">
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Precio simulado</div>
                        <div style="font-size:22px; font-weight:bold;">{eur2(simulated_price)}</div>
                        <div style="font-size:12px;">Base: {eur2(base_price)} · Cambio: {price_change_widget.value:+d}%</div>
                    </div>
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Ocupación simulada</div>
                        <div style="font-size:22px; font-weight:bold;">{pct_es(simulated_occupancy)}</div>
                        <div style="font-size:12px;">{simulated_tickets:,.0f} de {capacity:,.0f} plazas</div>
                    </div>
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Ocupación equilibrio</div>
                        <div style="font-size:22px; font-weight:bold;">{pct_es(break_even_occupancy) if np.isfinite(break_even_occupancy) else "No alcanzable"}</div>
                        <div style="font-size:12px;">{break_even_tickets:,.0f} pasajeros mínimos</div>
                    </div>
                    <div style="border:1px solid #ddd; border-radius:10px; padding:12px;">
                        <div style="color:#666;">Margen por salida</div>
                        <div style="font-size:22px; font-weight:bold;">{eur(margin)}</div>
                        <div style="font-size:12px;">Margen %: {pct_es(margin_pct)}</div>
                    </div>
                </div>
                <p style="margin-top:12px;"><b>Precio mínimo con esta demanda:</b> {eur2(break_even_price) if np.isfinite(break_even_price) else "No calculable"} · <b>Elasticidad:</b> {elasticity:.2f} · <b>Índice competidor:</b> {price_index:.2f}</p>
                <p><b>Revenue vs actual:</b> {pct_es(revenue_delta)} · <b>Margen vs actual:</b> {pct_es(margin_delta)}</p>
                <p><b>Lectura ejecutiva:</b> {decision}</p>
            </div>
            '''

            display(HTML(cards_html))

            comparison = pd.DataFrame({
                "metric": ["Revenue", "Coste", "Margen"],
                "value": [revenue, total_cost, max(margin, 0)]
            })

            plt.figure(figsize=(9, 4))
            sns.barplot(
                data=comparison,
                x="metric",
                y="value"
            )
            plt.axhline(0, linestyle="--", linewidth=1)
            plt.title("Resultado económico del escenario")
            plt.xlabel("")
            plt.ylabel("€ por salida")
            plt.tight_layout()
            plt.show()

    for widget in [
        route_widget,
        price_change_widget,
        demand_factor_widget,
        fixed_cost_widget,
        variable_cost_widget
    ]:
        widget.observe(render_break_even_tool, names="value")

    display(
        widgets.VBox([
            widgets.HTML("<h3>Controles del simulador de punto de equilibrio</h3>"),
            route_widget,
            price_change_widget,
            demand_factor_widget,
            fixed_cost_widget,
            variable_cost_widget,
            be_output
        ])
    )

    render_break_even_tool()

except Exception as e:
    print("No se pudo cargar el simulador interactivo en este entorno.")
    print("Error:", e)

## Exportación de herramienta HTML de punto de equilibrio

Además del simulador dentro de Colab, generamos un HTML autónomo:

`dashboard/break_even_pricing_decision_tool.html`

Este archivo puede abrirse directamente en el navegador y sirve como herramienta visual para portfolio.

In [ ]:
# ============================================================
# BREAK-EVEN TOOL 3. EXPORTAR HTML AUTÓNOMO
# ============================================================

import json

dashboard_path = f"{base_path}/dashboard"
os.makedirs(dashboard_path, exist_ok=True)

html_routes = {}

for _, row in break_even_base.iterrows():
    html_routes[row["route"]] = {
        "label": row["route"],
        "capacity": float(row["avg_capacity"]),
        "price": float(row["avg_ticket_price"]),
        "occupancy": float(row["avg_occupancy"] * 100),
        "fixed": float(row["avg_fixed_cost"]),
        "variable": float(row["avg_variable_cost"]),
        "elasticity": float(row["elasticity_for_simulation"]),
        "competitor": float(row["avg_competitor_price"])
    }

html_routes_json = json.dumps(html_routes, ensure_ascii=False)

html_output = f"""<!doctype html>
<html lang='es'>
<head>
<meta charset='utf-8'>
<meta name='viewport' content='width=device-width, initial-scale=1'>
<title>Levante Ferries — Break-even Pricing Tool</title>
<style>
body{{font-family:Arial,sans-serif;background:#f3f7fb;color:#102a43;margin:0;padding:24px}}
.container{{max-width:1120px;margin:0 auto}}
.card{{background:#fff;border:1px solid #d9e2ec;border-radius:18px;padding:18px;margin-bottom:16px;box-shadow:0 14px 35px rgba(16,42,67,.08)}}
h1{{font-size:40px;letter-spacing:-.04em;margin:0 0 8px}}
p{{color:#627d98;line-height:1.5}}
.grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}}
.kpis{{display:grid;grid-template-columns:repeat(4,1fr);gap:14px}}
label{{font-weight:700;font-size:13px;color:#627d98}}
select,input{{width:100%;margin-top:6px}}
select{{padding:9px;border:1px solid #d9e2ec;border-radius:9px}}
.kpi{{background:#fff;border:1px solid #d9e2ec;border-radius:14px;padding:14px}}
.kpi span{{display:block;color:#627d98;font-size:13px}}
.kpi strong{{display:block;font-size:25px;margin:6px 0}}
.good{{color:#147d64}}.warn{{color:#b7791f}}.bad{{color:#b5473a}}
.track{{position:relative;height:34px;border-radius:8px;background:#e7edf3}}
.zone{{position:absolute;top:0;right:0;bottom:0;background:rgba(118,199,192,.5);border-radius:0 8px 8px 0}}
.threshold{{position:absolute;top:-8px;bottom:-8px;width:3px;background:#102a43;transform:translateX(-50%)}}
.current{{position:absolute;top:50%;width:18px;height:18px;border:4px solid white;border-radius:50%;background:#168aad;box-shadow:0 0 0 2px #168aad;transform:translate(-50%,-50%)}}
.axis{{display:flex;justify-content:space-between;margin-top:10px;color:#627d98;font-size:12px}}
@media(max-width:850px){{.grid,.kpis{{grid-template-columns:1fr}}}}
</style>
</head>
<body>
<div class='container'>
<div class='card'>
<h1>Break-even Pricing Tool</h1>
<p>Herramienta de punto de equilibrio para analizar precio, ocupación, elasticidad, costes y margen por ruta.</p>
</div>

<div class='card grid'>
<label>Ruta<select id='route'></select></label>
<label>Cambio de precio <b id='priceValue'></b><input id='priceChange' type='range' min='-25' max='30' step='1' value='0'></label>
<label>Demanda externa <b id='demandValue'></b><input id='demandFactor' type='range' min='70' max='135' step='1' value='100'></label>
<label>Capacidad <b id='capacityValue'></b><input id='capacity' type='range' min='100' max='1000' step='10'></label>
<label>Coste fijo <b id='fixedValue'></b><input id='fixed' type='range' min='5000' max='50000' step='500'></label>
<label>Coste variable <b id='variableValue'></b><input id='variable' type='range' min='0' max='35' step='0.5'></label>
<label>Elasticidad <b id='elasticityValue'></b><input id='elasticity' type='range' min='-220' max='-20' step='1'></label>
</div>

<div class='kpis'>
<div class='kpi'><span>Precio simulado</span><strong id='simPrice'></strong><small id='priceContext'></small></div>
<div class='kpi'><span>Ocupación simulada</span><strong id='simOcc'></strong><small id='tickets'></small></div>
<div class='kpi'><span>Ocupación equilibrio</span><strong id='beOcc'></strong><small id='beSeats'></small></div>
<div class='kpi'><span>Margen por salida</span><strong id='margin'></strong><small id='marginContext'></small></div>
</div>

<div class='card'>
<div style='display:flex;justify-content:space-between;gap:12px;margin-bottom:12px;color:#627d98'>
<span>Ocupación simulada <b id='currentLabel'></b></span><span>Equilibrio <b id='thresholdLabel'></b></span>
</div>
<div class='track'><span class='zone' id='zone'></span><span class='threshold' id='threshold'></span><span class='current' id='current'></span></div>
<div class='axis'><span>0%</span><span>25%</span><span>50%</span><span>75%</span><span>100%</span></div>
</div>

<div class='card' id='decision'></div>
</div>

<script>
const routes = {html_routes_json};
const $ = id => document.getElementById(id);
const euro = new Intl.NumberFormat('es-ES',{{style:'currency',currency:'EUR',maximumFractionDigits:0}});
const euro2 = new Intl.NumberFormat('es-ES',{{style:'currency',currency:'EUR',minimumFractionDigits:2,maximumFractionDigits:2}});
const pct = v => new Intl.NumberFormat('es-ES',{{style:'percent',maximumFractionDigits:1,signDisplay:'exceptZero'}}).format(v);

function populate(){{
  Object.keys(routes).sort().forEach(key => $('route').add(new Option(routes[key].label, key)));
}}

function loadRoute(){{
  const r = routes[$('route').value];
  $('capacity').value = Math.round(r.capacity);
  $('fixed').value = Math.round(r.fixed/500)*500;
  $('variable').value = r.variable;
  $('elasticity').value = Math.round(r.elasticity*100);
  update();
}}

function update(){{
  const r = routes[$('route').value];
  const priceChange = Number($('priceChange').value)/100;
  const demandFactor = Number($('demandFactor').value)/100;
  const capacity = Number($('capacity').value);
  const fixed = Number($('fixed').value);
  const variable = Number($('variable').value);
  const elasticity = Number($('elasticity').value)/100;

  const basePrice = r.price;
  const simPrice = basePrice*(1+priceChange);
  const currentTickets = capacity*(r.occupancy/100);
  const simTickets = Math.min(Math.max(currentTickets*demandFactor*Math.pow(simPrice/basePrice,elasticity),0),capacity);
  const simOcc = simTickets/capacity;
  const revenue = simTickets*simPrice;
  const cost = fixed + simTickets*variable;
  const margin = revenue - cost;
  const contribution = simPrice - variable;
  const beTickets = contribution > 0 ? fixed/contribution : Infinity;
  const beOcc = beTickets/capacity;
  const bePrice = simTickets > 0 ? variable + fixed/simTickets : Infinity;
  const priceIndex = r.competitor ? simPrice/r.competitor : 1;

  $('priceValue').textContent = pct(priceChange);
  $('demandValue').textContent = pct(demandFactor);
  $('capacityValue').textContent = Math.round(capacity)+' plazas';
  $('fixedValue').textContent = euro.format(fixed);
  $('variableValue').textContent = euro2.format(variable);
  $('elasticityValue').textContent = elasticity.toFixed(2);

  $('simPrice').textContent = euro2.format(simPrice);
  $('priceContext').textContent = 'Base '+euro2.format(basePrice)+' · índice competidor '+priceIndex.toFixed(2);
  $('simOcc').textContent = pct(simOcc);
  $('tickets').textContent = Math.round(simTickets).toLocaleString('es-ES')+' de '+Math.round(capacity).toLocaleString('es-ES')+' plazas';
  $('beOcc').textContent = isFinite(beOcc) ? pct(beOcc) : 'No alcanzable';
  $('beSeats').textContent = isFinite(beTickets) ? Math.ceil(beTickets)+' pasajeros mínimos · precio mínimo '+euro2.format(bePrice) : 'El precio no cubre el coste variable';
  $('margin').textContent = euro.format(margin);
  $('margin').className = margin >= 0 ? 'good' : 'bad';
  $('marginContext').textContent = euro.format(revenue)+' ingresos · '+euro.format(cost)+' coste';

  const capped = Math.min(Math.max(beOcc*100,0),100);
  $('currentLabel').textContent = (simOcc*100).toFixed(1)+'%';
  $('thresholdLabel').textContent = isFinite(beOcc) ? (beOcc*100).toFixed(1)+'%' : 'No alcanzable';
  $('current').style.left = (simOcc*100)+'%';
  $('threshold').style.left = capped+'%';
  $('zone').style.width = beOcc<=1 ? (100-capped)+'%' : '0%';

  let decisionClass = 'bad';
  let text = 'Escenario no recomendado: no alcanza el punto de equilibrio.';
  if(margin >= 0 && simOcc >= beOcc && priceIndex <= 1.12){{
    decisionClass = 'good';
    text = 'Escenario atractivo: supera equilibrio, mantiene una posición competitiva razonable y genera margen positivo.';
  }} else if(margin >= 0){{
    decisionClass = 'warn';
    text = 'Escenario rentable, pero requiere revisión de ocupación, margen o posición frente al competidor.';
  }}
  $('decision').innerHTML = `<b>Lectura ejecutiva:</b> <span class='${{decisionClass}}'>${{text}}</span><br><b>Precio mínimo para esta demanda:</b> ${{isFinite(bePrice)?euro2.format(bePrice):'No calculable'}} · <b>Elasticidad:</b> ${{elasticity.toFixed(2)}}`;
}}

$('route').addEventListener('change',loadRoute);
['priceChange','demandFactor','capacity','fixed','variable','elasticity'].forEach(id => $(id).addEventListener('input',update));
populate(); loadRoute();
</script>
</body>
</html>"""

html_path = f"{dashboard_path}/break_even_pricing_decision_tool.html"

with open(html_path, "w", encoding="utf-8") as file:
    file.write(html_output)

print("Herramienta HTML de punto de equilibrio exportada en:")
print(html_path)

## 18. Insights automáticos

Generamos conclusiones ejecutivas a partir de los resultados.

In [ ]:
# ============================================================
# 27. INSIGHTS AUTOMÁTICOS
# ============================================================

most_inelastic_route = route_diagnosis.sort_values("elasticity_for_simulation", ascending=False).iloc[0]
most_elastic_route = route_diagnosis.sort_values("elasticity_for_simulation", ascending=True).iloc[0]
best_revenue_opportunity = route_diagnosis.sort_values("best_revenue_uplift_pct", ascending=False).iloc[0]
best_margin_opportunity = route_diagnosis.sort_values("best_margin_uplift_pct", ascending=False).iloc[0]

print("INSIGHTS EJECUTIVOS — ELASTICIDAD Y PRICING")
print("-" * 70)

print(
    f"1. La ruta más inelástica es {most_inelastic_route['route']} "
    f"con una elasticidad de {most_inelastic_route['elasticity_for_simulation']:.2f}."
)

print(
    f"2. La ruta más elástica es {most_elastic_route['route']} "
    f"con una elasticidad de {most_elastic_route['elasticity_for_simulation']:.2f}."
)

print(
    f"3. La mayor oportunidad simulada de revenue aparece en {best_revenue_opportunity['route']} "
    f"con el escenario {best_revenue_opportunity['best_revenue_scenario']} "
    f"y un uplift estimado de {best_revenue_opportunity['best_revenue_uplift_pct']:.2%}."
)

print(
    f"4. La mayor oportunidad simulada de margen aparece en {best_margin_opportunity['route']} "
    f"con el escenario {best_margin_opportunity['best_margin_scenario']} "
    f"y un uplift estimado de {best_margin_opportunity['best_margin_uplift_pct']:.2%}."
)

print("\nRecomendaciones por ruta:")
for _, row in route_diagnosis.iterrows():
    print(f"- {row['route']}: {row['recommendation']}")

## 19. Guardado de resultados

Exportamos todos los resultados en:

- CSV estándar.
- CSV compatible con Excel España.
- XLSX.

In [ ]:
# ============================================================
# 28. GUARDADO DE TABLAS DE ANÁLISIS
# ============================================================

tables_to_export = {
    "elasticity_simple_by_route": simple_elasticity,
    "elasticity_adjusted_by_route": adjusted_elasticity,
    "elasticity_comparison_by_route": elasticity_comparison,
    "elasticity_route_season": route_season_elasticity,
    "pricing_scenario_simulation_by_route": scenario_simulation,
    "pricing_best_scenarios_by_route": best_scenarios,
    "pricing_elasticity_route_diagnosis": route_diagnosis
}

for file_name, table in tables_to_export.items():
    export_table_files(
        dataframe=table,
        output_folder=reports_path,
        file_name=file_name,
        sheet_name=file_name[:31]
    )

print("Tablas de elasticidad y simulación exportadas correctamente.")

## 20. Resumen ejecutivo en Markdown

Creamos un archivo de resumen ejecutivo que podrá utilizarse para documentación del portfolio.

In [ ]:
# ============================================================
# 29. RESUMEN EJECUTIVO EN MARKDOWN
# ============================================================

executive_summary = f"""
# Executive Summary — Price Elasticity Analysis

## Project context

This notebook analyzes price-demand elasticity for a simulated ferry operator: Levante Ferries.

The goal is to understand how demand reacts to price changes and to identify opportunities for controlled price increases, promotional actions or price monitoring.

## General KPIs

- Total trips analyzed: {total_trips:,.0f}
- Tickets sold: {total_tickets:,.0f}
- Total revenue: {total_revenue:,.2f} €
- Total margin: {total_margin:,.2f} €
- Average occupancy: {avg_occupancy:.2%}
- Average ticket price: {avg_ticket_price:.2f} €
- Average margin percentage: {avg_margin_pct:.2%}

## Key elasticity findings

- Most inelastic route: {most_inelastic_route['route']} with elasticity {most_inelastic_route['elasticity_for_simulation']:.2f}
- Most elastic route: {most_elastic_route['route']} with elasticity {most_elastic_route['elasticity_for_simulation']:.2f}
- Best simulated revenue opportunity: {best_revenue_opportunity['route']} under scenario {best_revenue_opportunity['best_revenue_scenario']} with uplift {best_revenue_opportunity['best_revenue_uplift_pct']:.2%}
- Best simulated margin opportunity: {best_margin_opportunity['route']} under scenario {best_margin_opportunity['best_margin_scenario']} with uplift {best_margin_opportunity['best_margin_uplift_pct']:.2%}

## Business interpretation

Price elasticity should not be interpreted in isolation.  
A pricing recommendation should combine:

- Occupancy.
- Elasticity.
- Margin.
- Competitive price index.
- Seasonality.
- Revenue and margin simulation.

Routes with high occupancy, healthy margin and low price sensitivity are candidates for controlled price increases.  
Routes with low occupancy and high elasticity may require promotional actions or demand generation initiatives.

## Next step

The next notebook will build a demand prediction model to estimate expected tickets sold under different operational and commercial conditions.
"""

summary_path = f"{reports_path}/executive_summary_notebook_03.md"

with open(summary_path, "w", encoding="utf-8") as file:
    file.write(executive_summary)

print("Resumen ejecutivo guardado en:")
print(summary_path)

In [ ]:
# ============================================================
# 30. ARCHIVOS GENERADOS
# ============================================================

print("Imágenes generadas en el Notebook 3:")
for file in sorted(os.listdir(images_path)):
    if file.endswith(".png") and file.startswith(("21_", "22_", "23_", "24_", "25_", "26_", "27_")):
        print("-", file)

print("\nReportes generados en el Notebook 3:")
for file in sorted(os.listdir(reports_path)):
    if (
        file.startswith("elasticity_") or
        file.startswith("pricing_scenario_") or
        file.startswith("pricing_best_") or
        file.startswith("pricing_elasticity_") or
        file == "executive_summary_notebook_03.md"
    ):
        print("-", file)

## Conclusiones del Notebook 3

En este notebook se ha analizado la elasticidad precio-demanda de las rutas de ferry.

El análisis ha permitido:

- Estimar una elasticidad simple por ruta.
- Estimar una elasticidad ajustada con variables de control.
- Comparar la elasticidad estimada con la elasticidad simulada del dataset.
- Clasificar rutas como elásticas o inelásticas.
- Simular escenarios de cambios de precio.
- Estimar impacto en tickets vendidos, revenue y margen.
- Generar un diagnóstico ejecutivo por ruta.

La principal conclusión es que una recomendación de pricing no debe basarse solo en demanda o solo en elasticidad.

Una buena decisión de pricing debe combinar:

- Ocupación.
- Elasticidad.
- Margen.
- Precio frente al competidor.
- Estacionalidad.
- Simulación de revenue y margen.

Este notebook prepara la base para el Notebook 4, donde construiremos un modelo predictivo de demanda.